# Headless Video processing: Nvidia AI IOT NanoSAM
### .onnx (onnxruntime-gpu)
July 29th

NanoSAM is a Segment Anything (SAM) model variant that is capable of running in 🔥 real-time 🔥 on NVIDIA Jetson Orin Platforms with NVIDIA TensorRT.

In [ ]:
# ------------------------------------------------------------ #
#                 Clean environment setup
# ------------------------------------------------------------ #

"""
# clean env wipe & activate
conda env remove --name nanosam_arm64 -y
conda deactivate
conda create -n nanosam_arm64 python=3.10 -y
conda activate nanosam_arm64

sudo apt update
sudo apt install libcudnn8 libcudnn8-dev
conda install -c conda-forge libstdcxx-ng=12 -y

"""

In [ ]:
# ------------------------------------------------------------ #
#        Installation of nanosam_arm64 repo requirements
# ------------------------------------------------------------ #

"""
git clone https://github.com/NVIDIA-AI-IOT/nanosam
cd nanosam
python3 setup.py develop --user


# More requirements
pip install timm==1.0.17 ultralytics==8.0.120 matplotlib==3.5.3 'opencv-python<4.9' jetson-stats


# Typical np version mismatch
pip uninstall -y numpy
pip install "numpy>=1.19.2,<2.0"
# pip install numpy==1.26.4 

# Now install the CUDA PyTorch 2.5.0a0 (compatible with jp61 torch)
pip install --no-cache https://developer.download.nvidia.com/compute/redist/jp/v61/pytorch/torch-2.5.0a0+872d972e41.nv24.08.17622132-cp310-cp310-linux_aarch64.whl

# download torchvision 0.20.0a0 (compatible with jp61 torch)
cd ~/vision/
python3 setup.py install
cd ~

# --- ONNXRUNTIME-GPU (find appropriate aarch64 whl online) ------------- #

pip uninstall onnxruntime -y
pip install https://github.com/ultralytics/assets/releases/download/v0.0.0/onnxruntime_gpu-1.20.0-cp310-cp310-linux_aarch64.whl
pip install sympy==1.13.1 numpy==1.26.4

"""

# Debug Support

# Check your conda env GLIBCXX versions
# strings /home/copter/miniconda3/envs/nanosam_arm64/lib/python3.10/site-packages/zmq/backend/cython/../../../../.././libstdc++.so.6 | grep GLIBCXX

# trt2torch is apparently an issue and is obsolete for trt --version=10.3.0 
# additionally `pip install pycuda`

In [1]:
# from nanosam.utils.predictor import Predictor
import onnxruntime as ort


In [2]:
import onnx
import numpy as np
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import cv2
import os
import torch
import tensorrt     # import even if not used
from typing import Tuple, Optional, List
import time
from enum import Enum
from torchvision import transforms

In [3]:
# Verify versions

print(f"ONNX version: {onnx.__version__}")
print(f"NumPy version: {np.__version__}")       # shold be 1.26.4 or else causes mem core dump issues
print(f"TensorRT version: {tensorrt.__version__}")
print("Available ONNX Runtime providers: ", ort.get_available_providers())

ONNX version: 1.13.1
NumPy version: 1.26.4
TensorRT version: 10.3.0
Available ONNX Runtime providers:  ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


In [4]:
# 🧹 Clear the PyTorch CUDA cache
torch.cuda.empty_cache()

import gc
gc.collect()

# Prevent TensorRT plugin conflicts
os.environ['ORT_DISABLE_TRT_PLUGINS'] = '1'

# # Cleanup any existing sessions
# if 'enc_sess' in globals():
#     del enc_sess
# if 'dec_sess' in globals():
#     del dec_sess

In [5]:
# --- Resource allocation --------------
def check_gpu_memory():
    """Check GPU memory usage"""
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            memory_allocated = torch.cuda.memory_allocated(i) / 1024**3  # GB
            memory_reserved = torch.cuda.memory_reserved(i) / 1024**3    # GB
            memory_total = torch.cuda.get_device_properties(i).total_memory / 1024**3
            print(f"GPU {i}: {memory_allocated:.2f}GB allocated, {memory_reserved:.2f}GB reserved, {memory_total:.2f}GB total")
    else:
        print("CUDA not available")

def check_cpu_memory():
    """Check CPU memory usage"""
    memory = psutil.virtual_memory()
    process = psutil.Process()
    process_memory = process.memory_info().rss / 1024**3  # GB
    
    print(f"System RAM: {memory.used/1024**3:.2f}GB used / {memory.total/1024**3:.2f}GB total ({memory.percent:.1f}%)")
    print(f"Current process: {process_memory:.2f}GB")

def check_onnx_provider_status(enc_sess=None, dec_sess=None):
    """Check available ONNX Runtime providers and their status"""

    if enc_sess and enc_sess != None:
        print("Available Encoder ONNX Runtime providers:", enc_sess.get_providers())
        print("Available Decoder ONNX Runtime providers:", dec_sess.get_providers())
    else:
        available = ort.get_available_providers()
        print("Available ONNX Runtime providers:", available)

    
    # Check if TensorRT is causing issues
    import os
    trt_disabled = os.environ.get('ORT_DISABLE_TRT_PLUGINS', 'Not Set')
    print(f"ORT_DISABLE_TRT_PLUGINS: {trt_disabled}")


# check_cpu_memory()
# check_onnx_provider_status(enc_sess, dec_sess)

### Direct video processing

In [6]:
class PromptMode(Enum):
    POINT = "point"
    HEADING = "heading"
    BBOX = "bbox"
    EVERYTHING = "everything"

In [29]:
# Configuration Variables for the Notebook

ENCODER_MODEL="/home/copter/onnx_models/nvidia_ai_iot_resnet18_image_encoder.onnx"
DECODER_MODEL="/home/copter/onnx_models/nvidia_ai_iot_mobile_sam_mask_decoder.onnx"
INPUT_VIDEO = "/home/copter/Data/SimulatedDroneAngledCity - trimmed.mp4"
OUTPUT_VIDEO = "/home/copter/Data/sim_output.mp4"
GRID_SIZE = 8
SHOW_BBOX_OVERLAY = False
PROMPT_MODE = PromptMode.HEADING

In [ ]:
class NanoSAMEnhanced:
    def __init__(self, 
                 encoder_path: str,
                 decoder_path: str,
                 device_id: int = 0,
                 input_size: Tuple[int, int] = (1024, 1024)):
        """
        Enhanced NanoSAM with multiple prompt types, to be used for headless video processing
        """
        self.device_id = device_id
        self.input_size = input_size
        
        # Initialize ONNX Runtime sessions with proper configuration for Jetson
        providers = ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
        
        # Configure session options to avoid thread affinity warnings on ARM64
        sess_options = ort.SessionOptions()
        sess_options.inter_op_num_threads = 1  # Reduce inter-op parallelism
        sess_options.intra_op_num_threads = 4  # Set explicit thread count
        sess_options.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL
        sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
        
        # Suppress verbose logging
        sess_options.log_severity_level = 3  # Only show errors
        
        try:
            print("Loading encoder model...")
            self.encoder_session = ort.InferenceSession(encoder_path, sess_options, providers=providers)
            print("Loading decoder model...")
            self.decoder_session = ort.InferenceSession(decoder_path, sess_options, providers=providers)
             
            print(f"Using enc_sess providers: {self.encoder_session.get_providers()}")
            print(f"Using dec_sess providers: {self.decoder_session.get_providers()}")
        except Exception as e:
            print(f"❌ GPU session creation failed: {e}")
            print("🔄 Falling back to CPU-only providers...")
            try:
                # Fallback to CPU only
                cpu_providers = ['CPUExecutionProvider']
                self.encoder_session = ort.InferenceSession(encoder_path, sess_options=sess_options, providers=cpu_providers)
                self.decoder_session = ort.InferenceSession(decoder_path, sess_options=sess_options, providers=cpu_providers)
                print("✅ CPU fallback successful")
            except Exception as cpu_e:
                print(f"❌ CPU fallback also failed: {cpu_e}")
                raise RuntimeError("Failed to load ONNX models with any provider")

        # Get input/output names
        self.encoder_input_name = self.encoder_session.get_inputs()[0].name
        self.encoder_output_name = self.encoder_session.get_outputs()[0].name
        
        self.decoder_input_names = [inp.name for inp in self.decoder_session.get_inputs()]
        self.decoder_output_name = self.decoder_session.get_outputs()[0].name
        
        print(f"Decoder inputs: {self.decoder_input_names}")
        print(f"Decoder outputs: {[out.name for out in self.decoder_session.get_outputs()]}")
        
        # Check if we have multiple outputs (masks, scores, logits)
        self.decoder_outputs = [out.name for out in self.decoder_session.get_outputs()]
        if len(self.decoder_outputs) > 1:
            print(f"Multiple decoder outputs detected: {self.decoder_outputs}")
            # Look for mask-related output names
            mask_output_candidates = ['masks', 'low_res_masks', 'output_masks', 'segmentation_masks']
            for candidate in mask_output_candidates:
                if candidate in self.decoder_outputs:
                    self.decoder_output_name = candidate
                    print(f"Using mask output: {candidate}")
                    break
        
        # Image preprocessing
        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize(input_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                               std=[0.229, 0.224, 0.225])
        ])
        
        ## Initialize webcam
        # self.init_camera()

        # Video I/O  & FPS tracking
        self.cap = None
        self.video_writer = None
        self.fps_counter = 0
        self.fps_start_time = time.time()
        self.current_fps = 0
        
        # Prompt state
        self.current_mode = PromptMode.HEADING      # PromptMode.EVERYTHING
        self.positive_points = []
        self.negative_points = []
        self.fixed_points = []
        self.bbox_start = None
        self.bbox_end = None
        self.drawing_bbox = False
        self.everything_mode_active = False
        self.show_bbox = True                   # New toggle for bounding box visualization
        
        # Everything mode grid settings
        self.grid_size = 8   # 32  # Grid points for everything mode
        
        # Debug settings
        self.debug_mode = False
        self.save_masks = False
        self.frame_count = 0
        self.no_prompt_warning_counter = 0  # For throttling "no prompts" messages
    
    def init_camera(self):
        """Initialize camera for Logitech C925e"""
        # Method 1: Try V4L2 (usually best for USB cameras on Jetson)
        try:
            self.cap = cv2.VideoCapture(self.device_id, cv2.CAP_V4L2)
            if self.cap.isOpened():
                # Set optimal properties for C925e
                self.cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1920)
                self.cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 1080)
                self.cap.set(cv2.CAP_PROP_FPS, 30)
                self.cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc('M', 'J', 'P', 'G'))
                
                # Test if we can read a frame
                ret, _ = self.cap.read()
                if ret:
                    print(f"Initialized Logitech C925e with V4L2: /dev/video{self.device_id}")
                    print(f"Resolution: {int(self.cap.get(cv2.CAP_PROP_FRAME_WIDTH))}x{int(self.cap.get(cv2.CAP_PROP_FRAME_HEIGHT))}")
                    print(f"FPS: {self.cap.get(cv2.CAP_PROP_FPS)}")
                    return
        except Exception as e:
            print(f"V4L2 initialization failed: {e}")
        
        # Fallback methods...
        gst_pipeline = f'v4l2src device=/dev/video{self.device_id} ! image/jpeg,width=1920,height=1080,framerate=30/1 ! jpegdec ! videoconvert ! appsink drop=1'
        
        try:
            self.cap = cv2.VideoCapture(gst_pipeline, cv2.CAP_GSTREAMER)
            if self.cap.isOpened():
                print(f"Initialized camera with GStreamer: {gst_pipeline}")
                return
        except Exception as e:
            print(f"GStreamer initialization failed: {e}")
        
        # Basic OpenCV fallback
        self.cap = cv2.VideoCapture(self.device_id)
        if self.cap.isOpened():
            self.cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
            self.cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
            self.cap.set(cv2.CAP_PROP_FPS, 30)
            print(f"Initialized camera with basic OpenCV: /dev/video{self.device_id}")
        else:
            raise RuntimeError("Failed to initialize camera")
    
    def init_video_reader(self, input_video_path: str):
        """Initializes the video capture object."""
        self.cap = cv2.VideoCapture(input_video_path)
        if not self.cap.isOpened():
            raise IOError(f"❌ Cannot open video file: {input_video_path}")
        print(f"✅ Video file loaded: {input_video_path}")

    def init_video_writer(self, output_video_path: str, fps: float, frame_size: Tuple[int, int]):
        """Initializes the video writer object."""
        fourcc = cv2.VideoWriter_fourcc(*'mp4v') # Codec for .mp4 files
        self.video_writer = cv2.VideoWriter(output_video_path, fourcc, fps, frame_size)
        if not self.video_writer.isOpened():
            raise IOError(f"❌ Cannot create output video file: {output_video_path}")
        print(f"✅ Video writer initialized for: {output_video_path}")

    def preprocess_image(self, image: np.ndarray) -> np.ndarray:
        """Preprocess image for NanoSAM encoder"""
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        input_tensor = self.transform(image_rgb)
        input_batch = input_tensor.unsqueeze(0).numpy()
        return input_batch
    
    def encode_image(self, image: np.ndarray) -> np.ndarray:
        """Run image through encoder"""
        preprocessed = self.preprocess_image(image)
        encoder_outputs = self.encoder_session.run(
            [self.encoder_output_name],
            {self.encoder_input_name: preprocessed}
        )
        return encoder_outputs[0]
    
    def create_everything_prompts(self, frame_shape: Tuple[int, int]) -> Tuple[np.ndarray, np.ndarray]:
        """Create grid of points for everything mode"""
        h, w = frame_shape[:2]
        model_h, model_w = self.input_size
        self.grid_size = 5
        
        # Create grid of points
        x_points = np.linspace(start=0, stop=model_w-1, num=self.grid_size, dtype=int)
        y_points = np.linspace(start=0, stop=model_h-1, num=7, dtype=int)
        
        # Calculate y-coordinates for the upper 1/5th
        # upper_nth_height = model_h // 8
        upper_nth_height = len(y_points) // 3
    
        y_points = y_points[1:int(upper_nth_height)]
        # y_points = np.linspace(start=0, stop=upper_nth_height - 1, num=self.grid_size, dtype=int)
        
        print('points for everything mode... x: ', len(x_points), ', y: ', len(y_points))

        # append points in a grid
        points = []
        for y in y_points:
            for x in x_points:
                points.append([x, y])
        
        point_coords = np.array([points], dtype=np.float32)
        point_labels = np.ones((1, len(points)), dtype=np.float32)  # All positive
        
        return point_coords, point_labels

    def set_point_prompt(self, frame_shape: Tuple[int, int]) -> Tuple[Optional[np.ndarray], Optional[np.ndarray]]:
        """Create point prompts from clicked points"""
        h, w = frame_shape[:2]
        model_h, model_w = self.input_size
        
        px = w // 2
        py = h // 8

        self.fixed_points.append((px, py))

        points = []
        labels = []
        
        # Add scaled hardcoded points
        model_x = int(px * model_w / w)
        model_y = int(py * model_h / h)
        points.append([model_x, model_y])
        labels.append(1)
        
        if not points:
            return None, None
        
        point_coords = np.array([points], dtype=np.float32)
        point_labels = np.array([labels], dtype=np.float32)
        
        return point_coords, point_labels
    
    def create_point_prompts(self, frame_shape: Tuple[int, int]) -> Tuple[Optional[np.ndarray], Optional[np.ndarray]]:
        """Create point prompts from clicked points"""
        if not self.positive_points and not self.negative_points:
            return None, None
        
        h, w = frame_shape[:2]
        model_h, model_w = self.input_size
        
        points = []
        labels = []
        
        # Add positive points
        for px, py in self.positive_points:
            model_x = int(px * model_w / w)
            model_y = int(py * model_h / h)
            points.append([model_x, model_y])
            labels.append(1)
        
        # Add negative points
        for nx, ny in self.negative_points:
            model_x = int(nx * model_w / w)
            model_y = int(ny * model_h / h)
            points.append([model_x, model_y])
            labels.append(0)
        
        if not points:
            return None, None
        
        point_coords = np.array([points], dtype=np.float32)
        point_labels = np.array([labels], dtype=np.float32)
        
        return point_coords, point_labels
    
    def create_bbox_prompts(self, frame_shape: Tuple[int, int]) -> Tuple[Optional[np.ndarray], Optional[np.ndarray]]:
        """Create bounding box prompts"""
        if self.bbox_start is None or self.bbox_end is None:
            return None, None
        
        h, w = frame_shape[:2]
        model_h, model_w = self.input_size
        
        # Convert bbox to model coordinates
        x1 = min(self.bbox_start[0], self.bbox_end[0]) * model_w / w
        y1 = min(self.bbox_start[1], self.bbox_end[1]) * model_h / h
        x2 = max(self.bbox_start[0], self.bbox_end[0]) * model_w / w
        y2 = max(self.bbox_start[1], self.bbox_end[1]) * model_h / h
        
        # Convert bbox to corner points
        points = [
            [x1, y1],  # Top-left
            [x2, y2],  # Bottom-right
        ]
        
        point_coords = np.array([points], dtype=np.float32)
        point_labels = np.array([[2, 3]], dtype=np.float32)  # 2,3 for bbox corners
        
        return point_coords, point_labels

    def decode_masks(self, 
                    image_embeddings: np.ndarray,
                    point_coords: Optional[np.ndarray] = None,
                    point_labels: Optional[np.ndarray] = None,
                    show_best = False) -> List[np.ndarray]:
        """
        Decode masks from embeddings and prompts, filtering by score and surface area.
        Args:
            image_embeddings (np.ndarray): The image embeddings from the encoder. Shape: [batch_size, 256, 64, 64]
            point_coords (np.ndarray): Coordinates of point prompts. Shape: [batch_size, num_points, 2]
            point_labels (np.ndarray): Labels for point prompts. Shape: [batch_size, num_points]
            show_best (bool): If True, returns only the single highest-scoring mask.

        Return:
            all_detected_masks (List[np.ndarray]): List of masks that passed all filters.
                                                Each mask is a 2D np.ndarray.
        """
        
        if point_coords is None or point_labels is None:
            self.no_prompt_warning_counter += 1
            if self.no_prompt_warning_counter % 30 == 1:
                print("🔍 No prompts available")
            return []
        
        print(f"🔍 Decoding masks:")
        print(f"   Image embeddings shape: {image_embeddings.shape} [batch, channel, h, w]")
        print(f"   Point coords shape: {point_coords.shape} [batch, num_prompts, coords(x,y)]")
        print(f"   Point coords: {point_coords}")
        print(f"   Point labels shape: {point_labels.shape} [batch, num_prompts]")
        print(f"   Point labels: {point_labels}")
        
        # Prepare decoder inputs
        decoder_inputs = {
            'image_embeddings': image_embeddings,
            'point_coords': point_coords,
            'point_labels': point_labels,
        }
        
        # Add optional inputs with defaults
        if 'mask_input' in self.decoder_input_names:
            # Dummy mask input. Shape: [batch, 1, h, w]
            mask_input = np.zeros((1, 1, 256, 256), dtype=np.float32) 
            decoder_inputs['mask_input'] = mask_input
            print(f"   Added mask_input: {mask_input.shape}")
            
        if 'has_mask_input' in self.decoder_input_names:
            # Dummy flag to indicate no prior mask input. Shape: [1]
            decoder_inputs['has_mask_input'] = np.array([0], dtype=np.float32) 
            print(f"   Added has_mask_input: [0]")
        
        print(f"   Decoder input names: {list(decoder_inputs.keys())}")
        
        try:
            # Run the decoder to get all outputs.
            start_time = time.time()
            all_outputs = self.decoder_session.run(None, decoder_inputs)
            decode_time = time.time() - start_time
            print(f"✅ Decoder successful in {decode_time*1000:.2f}ms")

            # Assume the outputs are in the order [scores, masks].
            scores_out, masks_out = all_outputs
            
            # Print info about outputs for debugging.
            # Scores shape: [batch, num_masks]
            print(f"   Output 0 scores_out (iou_predictions): shape {scores_out.shape}, range [{scores_out.min():.3f}, {scores_out.max():.3f}]") 
            # Masks shape: [batch, num_masks, h, w]
            print(f"   Output 1 masks_out (low_res_masks): shape {masks_out.shape}, range [{masks_out.min():.3f}, {masks_out.max():.3f}]") 

            # Set filtering thresholds.
            confidence_threshold = 0.7
            area_min_ratio = 0.0005         # Filter out masks smaller than 0.05% of the image 
            area_max_ratio = 0.09           # Filter out masks larger than 50% of the image 

            # Get raw masks and scores, ensuring they are correctly shaped for iteration. 
            # Flattens to a 1D array of shape [num_masks] 
            scores = scores_out.flatten() 
            # Removes the batch dimension, leaving [num_masks, h, w] 
            masks_tensor = masks_out[0]
            print('unpacked masks and scores...')

            all_detected_masks = []
            
            # Determine original image dimensions for surface area calculation 
            # h_orig, w_orig = self.frame_shape[:2]
            h_orig, w_orig = self.frame_height, self.frame_width
            total_pixels = self.frame_height * self.frame_width

            # Get the index of the highest-scoring mask
            if show_best:
                print('shaowing best mask...')
                best_mask_idx = np.argmax(scores) 
                best_score = scores[best_mask_idx]
                best_mask = masks_tensor[best_mask_idx]
                print(f"   Selected best mask (index {best_mask_idx}) with score {best_score:.3f}")
                
                # Resize mask to original image size for surface area calculation
                best_mask_resized = cv2.resize(best_mask, (w_orig, h_orig))
                
                # Binarize mask for accurate area calculation (values above 0 are considered part of the object)
                binary_mask = (best_mask_resized > 0).astype(np.uint8) 
                area = np.sum(binary_mask)
                area_ratio = area / total_pixels
                print(f"   Best mask area ratio: {area_ratio:.4f}")
                
                # Check if best mask passes score and area filters
                if best_score >= confidence_threshold and area_min_ratio < area_ratio < area_max_ratio:
                    all_detected_masks.append(best_mask_resized)
                    print("   ✅ Best mask passed all filters.")
                else:
                    print("   ❌ Best mask failed one or more filters.")

            # Iterate through all masks and their corresponding scores.
            else:
                for i in range(len(scores)):
                    score = scores[i]
                    mask = masks_tensor[i]
                    
                    # Resize mask to original image size for area calculation
                    mask_resized = cv2.resize(mask, (w_orig, h_orig))
                    
                    # Binarize mask for accurate area calculation
                    binary_mask = (mask_resized > 0).astype(np.uint8)
                    area = np.sum(binary_mask)
                    area_ratio = area / total_pixels

                    # Filter masks based on both confidence and surface area
                    if score >= confidence_threshold and area_min_ratio < area_ratio < area_max_ratio:
                        print(f"   ✅ Accepted mask {i} with score {score:.3f} and area ratio {area_ratio:.4f}")
                        all_detected_masks.append(mask_resized)
                    else:
                        print(f"   ❌ Rejected mask {i} (score: {score:.3f}, area ratio: {area_ratio:.4f})")
            
            # Edge case: no masks found.
            if not all_detected_masks:
                print("   ⚠️ No masks passed the filters. Returning empty list.")
            
            return all_detected_masks
        except Exception as e:
            print(f"❌ Decoder failed: {e}")
            return np.zeros((1, 1, self.input_size[0], self.input_size[1]), dtype=np.float32)
    
    def update_fps(self):
        """Update FPS counter"""
        self.fps_counter += 1
        current_time = time.time()
        elapsed = current_time - self.fps_start_time
        
        if elapsed >= 1.0:
            self.current_fps = self.fps_counter / elapsed
            self.fps_counter = 0
            self.fps_start_time = current_time  
    
    def overlay_masks(self, image: np.ndarray, masks: List[np.ndarray], alpha: float = 0.5) -> np.ndarray:
        """Overlay multiple segmentation masks on image"""
        result = image.copy()
        
        print(f"🎨 Overlaying masks:")
        print(f"   Input image shape: {image.shape}")
        print(f"   Input is a list of {len(masks)} masks.") # Correct way to check the input
        
        h, w = image.shape[:2]
        
        # Colors for different masks
        colors = [
            [0, 255, 0],    # Green
            [255, 0, 0],    # Blue  
            [0, 0, 255],    # Red
            [255, 255, 0],  # Cyan
            [255, 0, 255],  # Magenta
            [0, 255, 255],  # Yellow
        ]
        
        mask_applied = False
        
        for i, mask in enumerate(masks):
            print(f"   Processing mask {i}: shape {mask.shape}, range [{mask.min():.3f}, {mask.max():.3f}]")
            
            if mask.shape != (h, w):
                mask_resized = cv2.resize(mask, (w, h))
                print(f"   Resized mask to: {mask_resized.shape}")
            else:
                mask_resized = mask
            
            # Try different thresholds to see what works
            # For SAM models, masks can have different value ranges
            thresholds = [0.0, 0.1, 0.3, 0.5]
            
            # Also try adaptive threshold based on mask statistics
            if mask_resized.max() > mask_resized.min():
                adaptive_thresh = mask_resized.mean() + 0.5 * mask_resized.std()
                thresholds.append(adaptive_thresh)
                print(f"   Added adaptive threshold: {adaptive_thresh:.3f}")
            
            for threshold in thresholds:
                mask_binary = (mask_resized > threshold).astype(np.uint8)
                pixel_count = np.sum(mask_binary)
                print(f"   Threshold {threshold:.3f}: {pixel_count} pixels ({pixel_count/(h*w)*100:.1f}%)")
                
                # Accept mask if it has reasonable coverage (0.1% to 50% of image)
                coverage_percent = pixel_count/(h*w)*100
                if 0.1 <= coverage_percent <= 50.0 and not mask_applied:
                    # Create colored overlay
                    color = colors[i % len(colors)]
                    overlay = result.copy()
                    overlay[mask_binary == 1] = color
                    
                    # Blend with result
                    result = cv2.addWeighted(result, 1 - alpha, overlay, alpha, 0)
                    mask_applied = True
                    print(f"   ✅ Applied mask {i} with threshold {threshold:.3f}, color {color}")
                    break
        
        if not mask_applied:
            print("   ⚠️  No masks were applied (all below threshold)")
        
        return result
    
    def overlay_bboxes(self, image: np.ndarray, masks: List[np.ndarray]) -> np.ndarray:
        """Overlay bounding boxes on segmented regions."""
        result = image.copy()
        
        print(f"📦 Overlaying bounding boxes:")
        print(f"   Input is a list of {len(masks)} masks.") # Correct way to check the input
        
        h, w = image.shape[:2]

        colors = [
            (0, 255, 0),    # Green
            (255, 0, 0),    # Blue  
            (0, 0, 255),    # Red
            (255, 255, 0),  # Cyan
            (255, 0, 255),  # Magenta
            (0, 255, 255),  # Yellow
        ]
        
        bbox_drawn = False
        
        # This loop correctly iterates over the list of masks
        for i, mask in enumerate(masks):
            # Check for empty masks before processing
            if mask.size == 0:
                print(f"   Skipping mask {i} as it is empty.")
                continue

            # Assuming the masks in the list are 2D arrays (H, W)
            if mask.shape != (h, w):
                mask_resized = cv2.resize(mask, (w, h), interpolation=cv2.INTER_LINEAR)
            else:
                mask_resized = mask
            
            mask_binary = (mask_resized > 0).astype(np.uint8) * 255 
            
            contours, _ = cv2.findContours(mask_binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            
            if not contours:
                print(f"   No contours found for mask {i}.")
                continue

            largest_contour = max(contours, key=cv2.contourArea)
            
            x, y, w_bbox, h_bbox = cv2.boundingRect(largest_contour)
            
            color = colors[i % len(colors)]
            cv2.rectangle(result, (x, y), (x + w_bbox, y + h_bbox), color, 2)
            bbox_drawn = True
            print(f"   ✅ Drawn bbox for mask {i} with color {color}")

        if not bbox_drawn:
            print("   ⚠️  No bounding boxes were drawn.")
        
        return result
    
    def return_bbox_xywh(self, image: np.ndarray, masks: List[np.ndarray]) -> np.ndarray:
        """Publish bounding boxes (xywh) of segmented mask regions."""
        result = image.copy()
        
        print(f"📦 Overlaying bounding boxes:")
        print(f"   Input is a list of {len(masks)} masks.") # Correct way to check the input
        
        h, w = image.shape[:2]

        colors = [
            (0, 255, 0),    # Green
            (255, 0, 0),    # Blue  
            (0, 0, 255),    # Red
            (255, 255, 0),  # Cyan
            (255, 0, 255),  # Magenta
            (0, 255, 255),  # Yellow
        ]
        
        bbox_drawn = False
        
        # This loop correctly iterates over the list of masks
        for i, mask in enumerate(masks):
            # Check for empty masks before processing
            if mask.size == 0:
                print(f"   Skipping mask {i} as it is empty.")
                continue

            # Assuming the masks in the list are 2D arrays (H, W)
            if mask.shape != (h, w):
                mask_resized = cv2.resize(mask, (w, h), interpolation=cv2.INTER_LINEAR)
            else:
                mask_resized = mask
            
            mask_binary = (mask_resized > 0).astype(np.uint8) * 255 
            
            contours, _ = cv2.findContours(mask_binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            
            if not contours:
                print(f"   No contours found for mask {i}.")
                continue

            largest_contour = max(contours, key=cv2.contourArea)
            
            x, y, w_bbox, h_bbox = cv2.boundingRect(largest_contour)
            
            color = colors[i % len(colors)]
            cv2.rectangle(result, (x, y), (x + w_bbox, y + h_bbox), color, 2)
            bbox_drawn = True
            print(f"   ✅ Drawn bbox for mask {i} with color {color}")

        if not bbox_drawn:
            print("   ⚠️  No bounding boxes were drawn.")
        
        return result
    
    def draw_prompts(self, image: np.ndarray) -> np.ndarray:
        """Draw current prompts on image"""
        result = image.copy()

        if self.current_mode == PromptMode.HEADING:
            px, py = self.fixed_points[0]
            cv2.circle(result, (px, py), 5, (0, 255, 0), -1)
            cv2.circle(result, (px, py), 7, (255, 255, 255), 2)

        # Draw positive points (green circles)
        for px, py in self.positive_points:
            cv2.circle(result, (px, py), 5, (0, 255, 0), -1)
            cv2.circle(result, (px, py), 7, (255, 255, 255), 2)
        
        # Draw negative points (red circles)
        for nx, ny in self.negative_points:
            cv2.circle(result, (nx, ny), 5, (0, 0, 255), -1)
            cv2.circle(result, (nx, ny), 7, (255, 255, 255), 2)
        
        # Draw bounding box
        if self.bbox_start and self.bbox_end:
            cv2.rectangle(result, self.bbox_start, self.bbox_end, (255, 0, 255), 2)
        elif self.bbox_start and self.drawing_bbox and self.bbox_end:
            cv2.rectangle(result, self.bbox_start, self.bbox_end, (128, 0, 128), 2)
        
        return result
    
    def clear_prompts(self):
        """Clear all prompts"""
        self.positive_points = []
        self.negative_points = []
        self.bbox_start = None
        self.bbox_end = None
        self.drawing_bbox = False
        self.everything_mode_active = False
        print("Cleared all prompts")
    
    def run(self):
        """DEPRECATED: Run video segmentation on .mp4"""
        print("Starting enhanced NanoSAM live segmentation...")
        
        cv2.namedWindow('NanoSAM Enhanced')
        cv2.setMouseCallback('NanoSAM Enhanced', self.mouse_callback)
        
        image_embeddings = None
        last_embedding_time = 0
        embedding_interval = 0.5
        inference_fps = 0.0
        inference_latency_ms = 0.0
        inference_start_time = None
        
        try:
            while True:
                ret, frame = self.cap.read()
                if not ret:
                    break
                
                current_time = time.time()
                
                # Update image embeddings periodically
                if current_time - last_embedding_time > embedding_interval:
                    image_embeddings = self.encode_image(frame)
                    last_embedding_time = current_time
                
                result_frame = frame.copy()
                
                # Get prompts based on current mode
                point_coords, point_labels = None, None
                    
                if self.current_mode == PromptMode.EVERYTHING and image_embeddings is not None:
                    point_coords, point_labels = self.create_everything_prompts(frame.shape)
                    # Visualize the grid points
                    if point_coords is not None:
                        h, w = frame.shape[:2]
                        model_h, model_w = self.input_size
                        for point in point_coords.squeeze(0).astype(int):
                            frame_x = int(point [0] * w / model_w)
                            frame_y = int(point [1] * h / model_h)
                            cv2.circle(result_frame, (frame_x, frame_y), 2, (255, 255, 0), -1) # Draw a small yellow circle
                elif self.current_mode == PromptMode.HEADING:
                    print("PromptMode.HEADING...")
                    point_coords, point_labels = self.set_point_prompt(frame.shape)
                elif self.current_mode == PromptMode.POINT:
                    point_coords, point_labels = self.create_point_prompts(frame.shape)
                elif self.current_mode == PromptMode.BBOX:
                    point_coords, point_labels = self.create_bbox_prompts(frame.shape)
                
                # Perform segmentation
                if point_coords is not None and image_embeddings is not None:

                    # --- Start Inference Timing ---
                    inference_start_time = time.time()

                    # This now returns a LIST of masks...
                    print(f"\n🚀 Running segmentation - Mode: {self.current_mode.value}")
                    masks_list = self.decode_masks(image_embeddings, point_coords, point_labels)
                    
                    # --- End Inference Timing ---
                    inference_time_s = time.time() - inference_start_time
                    inference_fps = 1.0 / inference_time_s if inference_time_s > 0 else 0
                    inference_latency_ms = inference_time_s * 1000

                    print(f"✅ Got {len(masks_list)} masks to display.")
                    
                    # The overlay functions must now handle a list of masks
                    if masks_list:
                        # The overlay functions need to be updated to accept a list
                        if self.show_bbox:
                            result_frame = self.overlay_bboxes(image=result_frame, masks=masks_list)
                        else:
                            result_frame = self.overlay_masks(image=result_frame, masks=masks_list)
                    else:
                        print(f"❌ No valid masks to display.")
                else:
                    if point_coords is None and self.no_prompt_warning_counter % 30 == 1:
                        print("🔍 No prompts available")
                        inference_fps = 0.0
                        inference_latency_ms = 0.0
                    if image_embeddings is None:
                        print("🔍 No image embeddings available")
                        inference_fps = 0.0
                        inference_latency_ms = 0.0
                
                # Draw prompts
                result_frame = self.draw_prompts(result_frame)
                
                # Update and display info
                self.update_fps()
                
                # Status text
                mode_text = f"Mode: {self.current_mode.value.upper()}"
                cv2.putText(result_frame, mode_text, (10, 30), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
                
                fps_text = f'FPS: {self.current_fps:.1f}'
                cv2.putText(result_frame, fps_text, (10, 60), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                
                inference_text = f'Inference: {inference_latency_ms:.1f} ms ({inference_fps:.1f} FPS)'
                cv2.putText(result_frame, inference_text, (10, 90), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
                
                # Point counts
                if self.current_mode == PromptMode.POINT:
                    point_text = f'Points: +{len(self.positive_points)} -{len(self.negative_points)}'
                    cv2.putText(result_frame, point_text, (10, 120), 
                               cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
                
                # Instructions
                instructions = [
                    "1:Point 2:BBox 3:Everything C:Clear D:Debug S:SaveMasks Q:Quit R:Reset"
                ]
                
                y_offset = result_frame.shape[0] - 20
                for instruction in instructions:
                    cv2.putText(result_frame, instruction, (10, y_offset), 
                               cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
                    y_offset -= 20
                
                cv2.imshow('NanoSAM Enhanced', result_frame)
        
        except KeyboardInterrupt:
            print("\nStopping...")
        finally:
            self.cleanup()
    
    def run_video_processing(self, input_video_path: str, output_video_path: str):
        """Processes video frame by frame and saves output."""
        print(f"Starting video segmentation on: {input_video_path}")

        self.init_video_reader(input_video_path)
        
        # Get video properties for writer
        fps = self.cap.get(cv2.CAP_PROP_FPS)
        width = int(self.cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(self.cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        frame_size = (width, height)
        
        self.init_video_writer(output_video_path, fps, frame_size)
        
        image_embeddings = None
        last_embedding_time = 0
        embedding_interval = 0.5

        try:
            while self.cap.isOpened():
                ret, frame = self.cap.read()
                if not ret:
                    break
                
                self.frame_count += 1
                current_time = time.time()
                
                # Update image embeddings periodically
                if current_time - last_embedding_time > embedding_interval:
                    image_embeddings = self.encode_image(frame)
                    last_embedding_time = current_time
                
                # Copy the frame
                result_frame = frame.copy()
                
                # Set the frame height and width params
                self.frame_height = result_frame.shape[0]
                self.frame_width = result_frame.shape[1]
                point_coords, point_labels = None, None
            
                # Get prompts based on current mode
                if self.current_mode == PromptMode.EVERYTHING and image_embeddings is not None:
                    point_coords, point_labels = self.create_everything_prompts(frame.shape)
                elif self.current_mode == PromptMode.HEADING:
                    point_coords, point_labels = self.set_point_prompt(frame.shape)
                    print(f'PromptMode set to == HEADING....')
                    print(f'prompt point set to {point_coords}')
                elif self.current_mode == PromptMode.POINT:
                    point_coords, point_labels = self.create_point_prompts(frame.shape)
                elif self.current_mode == PromptMode.BBOX:
                    point_coords, point_labels = self.create_bbox_prompts(frame.shape)

                # Draw prompts
                result_frame = self.draw_prompts(result_frame)

                # Perform segmentation
                if point_coords is not None and image_embeddings is not None:
                    masks_list = self.decode_masks(
                        image_embeddings=image_embeddings, 
                        point_coords=point_coords, 
                        point_labels=point_labels,
                        show_best=True
                    )
                    
                    # decide if to use masks or bboxes on the output video
                    if masks_list:
                        print(f'{len(masks_list)} masks returned....')
                        if self.show_bbox:
                            result_frame = self.overlay_bboxes(image=result_frame, masks=masks_list)
                        else:
                            result_frame = self.overlay_masks(image=result_frame, masks=masks_list)
                
                # Write the result frame to the output video
                self.video_writer.write(result_frame)
                
                print(f"Processed frame {self.frame_count}/{int(self.cap.get(cv2.CAP_PROP_FRAME_COUNT))}")

        except Exception as e:
            print(f"❌ An error occurred during processing: {e}")
        finally:
            print(f"Finished task...")
            self.cleanup()
            

    def cleanup(self):
        """Clean up resources"""
        if self.cap:
            self.cap.release()
        if self.video_writer:
            self.video_writer.release()
        cv2.destroyAllWindows()

In [39]:
# EXECUTION: NEW NOTEBOOK CELL

def main():
    """Main function to run the video segmentation script."""
    segmenter = NanoSAMEnhanced(
        encoder_path=ENCODER_MODEL,
        decoder_path=DECODER_MODEL,
    )
    
    # Configure the instance based on notebook variables
    segmenter.grid_size = GRID_SIZE
    segmenter.show_bbox = SHOW_BBOX_OVERLAY
    segmenter.current_mode = PROMPT_MODE
    
    # Run the processing
    segmenter.run_video_processing(INPUT_VIDEO, OUTPUT_VIDEO)

if __name__ == "__main__":
    main()

Loading encoder model...
Loading decoder model...
Using enc_sess providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
Using dec_sess providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
Decoder inputs: ['image_embeddings', 'point_coords', 'point_labels', 'mask_input', 'has_mask_input']
Decoder outputs: ['iou_predictions', 'low_res_masks']
Multiple decoder outputs detected: ['iou_predictions', 'low_res_masks']
Using mask output: low_res_masks
Starting video segmentation on: /home/copter/Data/SimulatedDroneAngledCity - trimmed.mp4
✅ Video file loaded: /home/copter/Data/SimulatedDroneAngledCity - trimmed.mp4
✅ Video writer initialized for: /home/copter/Data/sim_output.mp4


2025-08-21 17:07:35.865357459 [W:onnxruntime:Default, onnxruntime_pybind_state.cc:503 RegisterTensorRTPluginsAsCustomOps] The custom op domain name trt.plugins is already in session option.


PromptMode set to == HEADING....
prompt point set to [[[512. 128.]]]
🔍 Decoding masks:
   Image embeddings shape: (1, 256, 64, 64) [batch, channel, h, w]
   Point coords shape: (1, 1, 2) [batch, num_prompts, coords(x,y)]
   Point coords: [[[512. 128.]]]
   Point labels shape: (1, 1) [batch, num_prompts]
   Point labels: [[1.]]
   Added mask_input: (1, 1, 256, 256)
   Added has_mask_input: [0]
   Decoder input names: ['image_embeddings', 'point_coords', 'point_labels', 'mask_input', 'has_mask_input']
✅ Decoder successful in 28593.87ms
   Output 0 scores_out (iou_predictions): shape (1, 4), range [0.646, 0.782]
   Output 1 masks_out (low_res_masks): shape (1, 4, 256, 256), range [-14.675, 12.101]
unpacked masks and scores...
shaowing best mask...
   Selected best mask (index 2) with score 0.782
   Best mask area ratio: 0.0643
   ✅ Best mask passed all filters.
1 masks returned....
🎨 Overlaying masks:
   Input image shape: (1080, 1920, 3)
   Input is a list of 1 masks.
   Processing mask 